In [1]:
!pip install langchain
!pip install pinecone-client
!pip install pypdf
!pip install openai
!pip install tiktoken

In [2]:
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.vectorstores import Pinecone
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import os

In [3]:
my_file = PyPDFDirectoryLoader("pdfs")

In [4]:
data = my_file.load()

In [5]:
data[0]

Document(metadata={'source': 'pdfs\\YOLOv8_ State-of-the-Art Computer Vision Model.pdf', 'page': 0}, page_content='3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model\nhttps://yolov8.com 1/5\nWhat is YOLOv8?\nYOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.\nY ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA\nGPUs, and macOS systems with Roboflow Inference, an open source Python package for running\nvision models.\nGet Started Using YOLOv8\nRoboflow is the fastest way to get YOLOv8 running in production. Manage dataset versioning, preprocessing, augmentation,\ntraining, evaluation, and deployment all in one workflow . Easily upload data, train YOLOv8 with best-practice defaults,\ncompare runs, and deploy to edge, cloud, or API in minutes. T ry a  YOLOv8 model  on Roboflow with this workflow:\nYOLOv5 YOLOv8 YOLO1 1\nP y t h o n c U R L J a v a s c r i p t S w i f t . N e t\nfrom inference

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)

In [7]:
text_chunks = text_splitter.split_documents(data)

In [8]:
len(text_chunks)

16

In [9]:
print(text_chunks[0])

page_content='3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model
https://yolov8.com 1/5
What is YOLOv8?
YOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.
Y ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA
GPUs, and macOS systems with Roboflow Inference, an open source Python package for running
vision models.
Get Started Using YOLOv8' metadata={'source': 'pdfs\\YOLOv8_ State-of-the-Art Computer Vision Model.pdf', 'page': 0}


In [10]:
print(text_chunks[0].page_content)

3/4/26, 11:58 PM YOLOv8: State-of-the-Art Computer Vision Model
https://yolov8.com 1/5
What is YOLOv8?
YOLOv8 is a computer vision model architecture developed by Ultralytics, the creators of YOLOv5.
Y ou can deploy YOLOv8 models on a wide range of devices, including NVIDIA  Jetson, NVIDIA
GPUs, and macOS systems with Roboflow Inference, an open source Python package for running
vision models.
Get Started Using YOLOv8


In [11]:
import os
os.environ["OPENAI_API_KEY"]="sk-"

PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY',"X")
# PINECONE_API_ENV = os.environ.get('PINECONE_API_ENV',"gcp-starter")

In [12]:
embeddings = OpenAIEmbeddings()

<ipython-input-12-73ad2f8e367a>:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  embeddings = OpenAIEmbeddings()


In [13]:
# Import the Pinecone library
from pinecone import Pinecone

In [14]:
# Initialize a Pinecone client with your API key
pc = Pinecone(api_key="YOUR_API_KEY")

In [16]:
index = pc.Index('testing')

In [17]:
docsearch = pc.from_texts([t.page_content for t in text_chunks], embeddings, index_name=index_name)

In [18]:
query = "which models does the Yolov8 outperform"

In [19]:
docs = docsearch.similarity_search(query)

In [20]:
## the output here would be in the form of vector
docs

In [21]:
## we need to get output in form of text, so we will use LLM and Retrieval QA

In [22]:
llm = OpenAI()

<ipython-input-22-4ab60154b5e1>:1: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAI`.
  llm = OpenAI()


In [23]:
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=docsearch.as_retriever())

In [24]:
query = "which models does the Yolov8 outperform"

In [25]:
# this model will now generate correct output , v8 outperforms v7, v6, v5 etc
qa.run(query)

In [27]:
# we can now create a chatbot
import sys
while True:
    user_input = input("Input Prompt: ")
    if user_input == 'exit':
        print('Exiting..!!')
        sys.exit()
    if user_input == "":
        continue
    result = qa({'query':user_input})
    print(f"Answer: {result['result']}")